# Guide 1 — Match Setup & Configuration

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## What is this notebook about?

Before the board can play, it needs to know a few things: what web address the
referee computer is at, what team we are, and how big the game board is. Think of
it like typing your name and picking your color before starting a video game.

None of this is tricky code — it's mostly filling in the right values. But if one
value is wrong, nothing else works, so it's worth understanding each line.


### How this guide fits in

**Depends on:** nothing — start here (after the overview). **Used by:** every other guide.

*New here? Read **Guide 0 — How Everything Connects** first for the big picture.*


### Step 1: Load the helper package

This first block unzips a small helper and loads `pynqp2p`, the package that lets
our board talk to the referee over the network. You just run it — nothing to
change.


In [ ]:
import sys, zipfile

wheel = '/home/root/jupyter_notebooks/pynqp2p_pkg/vendor_wheels/getmac-0.9.5-py2.py3-none-any.whl'
extract_dir = '/home/root/jupyter_notebooks/pynqp2p_pkg/vendor_wheels/extracted'
zipfile.ZipFile(wheel).extractall(extract_dir)

sys.path.insert(0, extract_dir)
sys.path.insert(0, '/home/root/jupyter_notebooks/pynqp2p_pkg')

import pynqp2p
print('pynqp2p OK from', pynqp2p.__file__)

### Step 2: Find the model files

The board uses a pre-trained "eye" (a YOLO model) to recognize cards. This block
just finds those files on the board and remembers where they are.


In [ ]:
from pathlib import Path
import os
import sys
import time
import json
import re
import html
import base64
import subprocess
import requests
import threading
import queue
import colorsys
import random
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
from pynq_dpu import DpuOverlay
import pynqp2p

BOOTCAMP_301_DIR = Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 301 - Object Detection')
NOTEBOOK_DIR = Path.cwd()

if not (NOTEBOOK_DIR / 'tf_yolov3_voc.xmodel').exists() and BOOTCAMP_301_DIR.exists():
    NOTEBOOK_DIR = BOOTCAMP_301_DIR

MODEL_PATH = NOTEBOOK_DIR / 'tf_yolov3_voc.xmodel'
CLASSES_PATH = NOTEBOOK_DIR / 'img' / 'voc_classes.txt'

# Pre-game riddle solving reuses the shared halo Strix LLM helper from the
# ai_llm bootcamp notebook (server IP, model names, and chat plumbing all
# live there) -- see PYNQ_301-Memory_Game_Grid_Detection-LLM.ipynb.
AI_HELPER_DIR = Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/ai_llm')
if str(AI_HELPER_DIR) not in sys.path:
    sys.path.append(str(AI_HELPER_DIR))
import bootcamp_ai

print(f'Using notebook assets from: {NOTEBOOK_DIR}')

### Step 3: Set the board's clock

These boards forget the date every time they turn off (they have no battery
clock). This code lets you set the time. It matters mostly for making log messages
readable. You can set it by hand, or copy the time from another computer on the
network.


In [ ]:
def set_board_time(date_string):
    """Sets the board's system clock via `date -s`. Jupyter on these boards
    already runs as root, so no sudo is needed."""
    result = subprocess.run(['date', '-s', date_string], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Failed to set date: {result.stderr.strip()}')
    now = subprocess.run(['date'], capture_output=True, text=True).stdout.strip()
    print(f'Board time set to: {now}')


def sync_board_time_from_http(url):
    """Best-effort: derive the current time from an HTTP server's Date
    header instead of typing it in by hand -- point this at the broker, the
    master, or any other host reachable from the board."""
    import urllib.request
    import email.utils

    request = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(request, timeout=5) as response:
        date_header = response.headers.get('Date')
    if not date_header:
        raise RuntimeError(f'{url} did not return a Date header.')
    parsed = email.utils.parsedate_to_datetime(date_header)
    set_board_time(parsed.strftime('%Y-%m-%d %H:%M:%S'))


board_time_text = widgets.Text(value='', placeholder='YYYY-MM-DD HH:MM:SS', description='Set time:')
set_time_button = widgets.Button(description='Set Board Time', button_style='warning')
sync_time_url_text = widgets.Text(value='', placeholder='e.g. http://192.168.1.100:5000', description='Sync from:')
sync_time_button = widgets.Button(description='Sync From HTTP', button_style='info')
time_output = widgets.Output()


def on_set_time_clicked(_button):
    with time_output:
        clear_output(wait=True)
        try:
            if not board_time_text.value.strip():
                raise ValueError('Enter a date/time first, e.g. 2026-07-16 09:00:00')
            set_board_time(board_time_text.value.strip())
        except Exception:
            import traceback
            traceback.print_exc()


def on_sync_time_clicked(_button):
    with time_output:
        clear_output(wait=True)
        try:
            if not sync_time_url_text.value.strip():
                raise ValueError('Enter a reachable URL first, e.g. the broker address.')
            sync_board_time_from_http(sync_time_url_text.value.strip())
        except Exception:
            import traceback
            traceback.print_exc()


set_time_button.on_click(on_set_time_clicked)
sync_time_button.on_click(on_sync_time_clicked)

# Inserted into the single dashboard in Section 10 so controls never appear
# in a separate notebook output area.
board_clock_panel = widgets.VBox([
    widgets.HTML('<h3>Board Clock</h3>'),
    widgets.HTML('<small>Set the board\'s clock manually, or sync it from a reachable server\'s HTTP Date header.</small>'),
    widgets.HBox([board_time_text, set_time_button]),
    widgets.HBox([sync_time_url_text, sync_time_button]),
    time_output,
])

### Step 4: The important part — match settings

These are the values you change **before every match**. Read the comment on each
line. The RC team gives you these numbers.

- `SERVER`, `BROKER_KEY`, `REFEREE_ID` — where and how to connect (can change each
  round).
- `MASTER_ID`, `TEAM_SECRET` — given once when you register; they stay the same.
- `TEAM_NAME` — must be spelled **exactly** like you registered.
- `GRID_ROWS`, `GRID_COLS` — how many rows and columns the real board has.

The board positions are named like `B3` (row letter + column number), which is
exactly how the referee talks about them too.


In [ ]:
SERVER = '192.168.101.5:35050'      # RC team gives you this
BROKER_KEY = 'bootcamp2026'        # RC team gives you this
REFEREE_ID = 'arena-1-referee'     # RC team gives you this -- may change between rounds
MASTER_ID = 'master-referee'       # RC team gives you this -- does NOT change between rounds
TEAM_NAME = 'red'                # your team name, exactly as registered

# Shown to your team once at registration. Leave blank to skip self-reporting your
# MAC -- the operator can still enter it for you manually, this just saves a step.
TEAM_SECRET = 'Kduowa9A'

# Leave blank to use this board's real MAC address (pynqp2p.get_id()).
# Only override this for lab testing on a machine with no eth0.
BOARD_ID_OVERRIDE = ''

GRID_ROWS = 5   # lettered A..E
GRID_COLS = 6   # numbered 1..6

# Example for an angled camera -- top-left, top-right, bottom-right, bottom-left:
# BOARD_CORNERS = np.float32([[80, 60], [1180, 80], [1220, 690], [60, 700]])
BOARD_CORNERS = None

# Fixed for this notebook -- see PYNQ_302-<approach>-<color>.ipynb for the
# other three approaches. Re-derived authoritatively by
# set_detection_approach(DETECTION_APPROACH) at the end of the ArUco/dispatch
# cell below; the DETECTION_MODE/CARD_POSITION_MODE values here are just for
# a reader's reference before that cell runs.
DETECTION_APPROACH = 'aruco_border'
DETECTION_MODE = 'grid'
CARD_POSITION_MODE = 'yolo_center'

### Check yourself

1. Which two settings stay the same every round, and which ones can change?
2. If you spelled `TEAM_NAME` wrong, would you get an error message? (No — that's
   why it's easy to miss!)
